# 3.1 Learn a classifier without leaking geography

**Learning objectives:** fit a simple embedding classifier; hold out spatial groups; compare a baseline; recognize annual-data limitations.

**How to learn:** predict - run - explain - change one thing. Spend 45–90 minutes here.

**Data:** the default worked example is deterministic synthetic practice data near Colorado. It is not SNOTEL, a watershed survey, a burn map, or a foundation-model output.


## Before you run

An embedding is a learned vector, not a named physical band. Nearby pixels are often correlated, so random pixel splits can exaggerate generalization. Fit scaling on training data only. Our simulated vectors are not AlphaEarth embeddings.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/"course.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course import OUT, points


## Worked example: spatial holdout


In [ ]:
from course import embeddings, evaluate
frame=points()
x=embeddings(frame,64)
model,metrics,confusion,train,test=evaluate(frame,x)
assert set(frame.block.iloc[train]).isdisjoint(frame.block.iloc[test])
print(metrics)
print('Confusion matrix (rows=true, columns=predicted):\n',confusion)
frame['probability']=model.predict_proba(x)[:,list(model.classes_).index(1)]
frame['split']=np.where(frame.index.isin(test),'test','train')
frame.to_csv(OUT/'synthetic_predictions.csv',index=False)
plt.scatter(frame.longitude,frame.latitude,c=frame.probability,vmin=0,vmax=1)
plt.colorbar(label='Synthetic class-1 score');plt.title('Practice surface; includes training locations');plt.show()


## Label-budget experiment

Use the same held-out points for every budget. These are learning diagnostics; selecting a final model requires a separate validation set.


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
rows=[]
for seed in range(5):
    order=np.random.default_rng(seed).permutation(train)
    for budget in [16,32,64,len(train)]:
        subset=order[:budget]
        if frame.label.iloc[subset].nunique()<2: continue
        candidate=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000))
        candidate.fit(x[subset],frame.label.iloc[subset])
        rows.append(dict(seed=seed,labels=len(subset),score=balanced_accuracy_score(frame.label.iloc[test],candidate.predict(x[test]))))
curve=pd.DataFrame(rows).groupby('labels').score.agg(['mean','std'])
print(curve)
curve['mean'].plot(yerr=curve['std'],marker='o',ylim=(0,1),ylabel='Held-out balanced accuracy');plt.show()


## Your turn

Replace the group split with a random split and compare results. Increase embedding noise. Does doubling labels reliably improve all five seeds?

Use a new cell below to try this before opening the solution.


In [ ]:
# Write your experiment here. The worked example above is already complete.


<details><summary>Solution and reasoning - open after trying</summary>

A random split answers a different question: interpolation among nearby samples. It may or may not score higher in this fixture. Noise usually reduces performance; finite samples can make learning curves non-monotonic. Report variation rather than claiming a plateau from one run.

</details>


## Check yourself

Expected: disjoint blocks, a 2×2 confusion matrix, and a saved synthetic prediction table. What would count as independent labels? Why is 2021 annual imagery unsuitable for isolating a late-December event?

**Exit ticket:** write one result, one explanation, and one limitation. Save the notebook with your own observations.


## Transfer to real data

The optional acquisition script in `scripts/fetch_embeddings.py` uses the documented `GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL` collection and 64 A-bands. Follow `docs/REAL_DATA.md` to supply independently interpreted labels and an Earth Engine project. Use 2022 for a post-2021 annual comparison, while acknowledging that an annual embedding does not isolate fire effects. Run `scripts/compare_embeddings.py` after acquiring both sources.
